In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dask.distributed import Client as DaskClient
from odc.geo.geom import BoundingBox
from odc.stac import configure_s3_access, load
from pystac_client import Client
import os

In [4]:
# Configure S3 access in Dask and outside
configure_s3_access(aws_unsigned=True)

# Set up a Dask client
dask_client = DaskClient(
    n_workers=2,
    threads_per_worker=16,
    memory_limit="30GB",
)

In [5]:
catalog = "https://stac.staging.digitalearthpacific.io"
client = Client.open(catalog)

In [6]:
# Bounding box 

# TRAINING AREA 
# over Viti Levu, Fiji : left=177.2, bottom=-18.3, right=178.8, top=-17.2
# over Rarotonga, Cook Island : left=-159.85, bottom=-21.28, right=-159.7, top=-21.19
# over Majuro, Marshall Island : left=171, bottom=7.05, right=171.4, top=7.25
# over Ngerulmud, Palau Island : left=134.2, bottom=7.11, right=134.66, top=7.75

# TESTING AREA 
# over Vanua Levu, Fiji (label available) : left=178.55, bottom=-17.2, right=179.98, top=-15.85
# over Nouméa, New-Calédonia (no label) : left=166.35, bottom=-22.33, right=166.56, top=-22.16
# Labasa Fiji (no label) : left=179.1, bottom=-16.6, right=179.45, top=-16.3 

bbox = BoundingBox(left=177.2, bottom=-18.3, right=178.8, top=-17.2)
bbox.explore()

# Sentinel 2 

In [7]:
items = client.search(
    collections=["dep_s2_geomad"],
    bbox=bbox,
    datetime="2024",
).item_collection()

# Load the data
chunks = dict(x=2048, y=2048)
data = load(
    items,
    chunks=chunks,
    measurements=["coastal", "blue", "green", "red","rededge1", "rededge2",  "rededge3", "nir", "nir08",   "nir09",  "swir16",  "swir22"],
)
spectral = data.where(data != 0, np.nan)

print(spectral)

<xarray.Dataset> Size: 53GB
Dimensions:      (time: 1, y: 19200, x: 28800)
Coordinates:
  * y            (y) float64 154kB -1.888e+06 -1.888e+06 ... -2.08e+06 -2.08e+06
  * x            (x) float64 230kB 2.952e+06 2.952e+06 ... 3.24e+06 3.24e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2024-01-01
Data variables:
    coastal      (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge1     (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge2     (time, y, x) float64 4GB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    rededge3     (time, y, x) float64 4GB dask.array<chunksize=(1, 2048,

In [8]:
rename_dict = {
    "coastal": "COASTAL_AEROSOL",
    "blue": "BLUE",
    "green": "GREEN",
    "red": "RED",
    "rededge1": "RED_EDGE_1",
    "rededge2": "RED_EDGE_2",
    "rededge3": "RED_EDGE_3",
    "nir": "NIR_BROAD",   # check carefully!
    "nir08": "NIR_NARROW",
    "nir09": "CIRRUS",
    "swir16": "SWIR_1",
    "swir22": "SWIR_2",
}
ds = spectral.rename(rename_dict)
ds.to_netcdf("./data_FM/s2/s2_2024.nc")

In [7]:
patch_file = "./data_FM/s2/s2_2024.zarr"
spectral.to_zarr(patch_file, mode="w")

## Label

In [9]:
gdf = gpd.read_file("./data/label/lulc_fiji.gpkg").to_crs(spectral.odc.crs)

from sklearn.model_selection import train_test_split


gdf_train, gdf_temp = train_test_split(
    gdf, test_size=0.30, random_state=42, shuffle=True
)

gdf_val, gdf_test = train_test_split(
    gdf_temp, test_size=0.50, random_state=42, shuffle=True
)

print("Train:", len(gdf_train))
print("Val:", len(gdf_val))
print("Test:", len(gdf_test))

Train: 7486
Val: 1604
Test: 1605


/srv/conda/envs/notebook/lib/python3.11/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'lulc_fiji.gpkg': 'fj_lulc_data_points_merged__fj_lulc' (default), 'lulc fiji'. Specify layer parameter to avoid this warning.
  result = read_func(


In [10]:
gdf_train.to_csv('./data_FM/label/train.csv', index=False)  
gdf_val.to_csv('./data_FM/label/val.csv', index=False)  
gdf_test.to_csv('./data_FM/label/test.csv', index=False)  

In [8]:
gdf = gpd.read_file("./data/label/lulc_fiji.gpkg").to_crs(spectral.odc.crs)
gdf.to_csv('./data_FM/label/label.csv', index=False)  

/srv/conda/envs/notebook/lib/python3.11/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'lulc_fiji.gpkg': 'fj_lulc_data_points_merged__fj_lulc' (default), 'lulc fiji'. Specify layer parameter to avoid this warning.
  result = read_func(


In [9]:
print(spectral.odc.crs)

PROJCRS["WGS 84 / PDC Mercator",BASEGEOGCRS["WGS 84",ENSEMBLE["World Geodetic System 1984 ensemble",MEMBER["World Geodetic System 1984 (Transit)"],MEMBER["World Geodetic System 1984 (G730)"],MEMBER["World Geodetic System 1984 (G873)"],MEMBER["World Geodetic System 1984 (G1150)"],MEMBER["World Geodetic System 1984 (G1674)"],MEMBER["World Geodetic System 1984 (G1762)"],MEMBER["World Geodetic System 1984 (G2139)"],MEMBER["World Geodetic System 1984 (G2296)"],ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]],ENSEMBLEACCURACY[2.0]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]],ID["EPSG",4326]],CONVERSION["Pacific Disaster Center Mercator",METHOD["Mercator (variant A)",ID["EPSG",9804]],PARAMETER["Latitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8801]],PARAMETER["Longitude of natural origin",150,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Scale factor at natural origin",1,SCALEUNIT["unity",1],ID["EPSG",8805]],PA